In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

In [ ]:
data_dir = Path("/home/xgupke/Documents/projects/EmmaElinTrombocytes/datafiles/images")
labels_dir = data_dir / "labels"
images_dir = data_dir / "rare_granules"
csv_name = "rare_granules_raw_correct.csv"

model_dir = '/home/xgupke/Documents/projects/EmmaElinTrombocytes/pixel_classifier'
model_dir = Path(model_dir)

run = 1
run = str(run)
model_dir = model_dir / "runs" / run
model_path = model_dir / "models_px" / "random_forest_px.joblib"

In [ ]:
labels_dict = {
    1: "granule",
    2: "tail",
    3: "spike"
}

In [ ]:
rf = joblib.load(model_path)

In [ ]:
import json

with open(model_dir / "pixel_feature_names.json", "r") as f:
    feature_names = json.load(f)

print(feature_names)

['intensity', 'gaussian_sigma_5.93', 'sobel_sigma_5.93', 'LoG_sigma_5.93', 'hessian_eig1_sigma_5.93', 'hessian_eig2_sigma_5.93', 'gaussian_sigma_11.85', 'sobel_sigma_11.85', 'LoG_sigma_11.85', 'hessian_eig1_sigma_11.85', 'hessian_eig2_sigma_11.85', 'gaussian_sigma_20.74', 'sobel_sigma_20.74', 'LoG_sigma_20.74', 'hessian_eig1_sigma_20.74', 'hessian_eig2_sigma_20.74']


In [ ]:
# obtain X_test and y_test

with open(model_dir / "test_images.txt", "r") as f:
    test_images = [line.strip() for line in f if line.strip()]

feature_map = np.load(model_dir / "pixel_features.npy")

# Load row ranges
with open(model_dir / "pixel_feature_row_ranges.json", "r") as f:
    row_ranges = json.load(f)

# Example:
# image_names = ["40135_E1_x4300_-10um_0013.png", ...]

selected_rows = []

for name in test_images:
    start = row_ranges[name]["start"]
    end = row_ranges[name]["end"]

    selected_rows.append(feature_map[start:end])

# Combine all selected rows into one array
selected_features = np.concatenate(selected_rows, axis=0)

print(selected_features.shape, feature_map.shape)


(15063126, 17) (74322005, 17)


In [ ]:
X_test = selected_features[:,:-1]
y_test = selected_features[:,-1]

del selected_features, feature_map

In [ ]:
feature_names

['intensity',
 'gaussian_sigma_5.93',
 'sobel_sigma_5.93',
 'LoG_sigma_5.93',
 'hessian_eig1_sigma_5.93',
 'hessian_eig2_sigma_5.93',
 'gaussian_sigma_11.85',
 'sobel_sigma_11.85',
 'LoG_sigma_11.85',
 'hessian_eig1_sigma_11.85',
 'hessian_eig2_sigma_11.85',
 'gaussian_sigma_20.74',
 'sobel_sigma_20.74',
 'LoG_sigma_20.74',
 'hessian_eig1_sigma_20.74',
 'hessian_eig2_sigma_20.74']

In [ ]:
importances = rf.feature_importances_

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print('Feature importance')
print(feature_importance)

Feature importance
                     feature  importance
9   hessian_eig1_sigma_11.85    0.163972
14  hessian_eig1_sigma_20.74    0.128075
7          sobel_sigma_11.85    0.086990
2           sobel_sigma_5.93    0.076186
13           LoG_sigma_20.74    0.074029
10  hessian_eig2_sigma_11.85    0.067050
8            LoG_sigma_11.85    0.066550
12         sobel_sigma_20.74    0.055591
11      gaussian_sigma_20.74    0.052422
5    hessian_eig2_sigma_5.93    0.046872
4    hessian_eig1_sigma_5.93    0.040243
6       gaussian_sigma_11.85    0.037984
1        gaussian_sigma_5.93    0.035300
15  hessian_eig2_sigma_20.74    0.030551
3             LoG_sigma_5.93    0.021020
0                  intensity    0.017166


In [ ]:
# THIS IS A RAM KILLER

# from sklearn.inspection import permutation_importance

# result = permutation_importance(
#     rf,
#     X_test,
#     y_test,
#     n_repeats=10,
#     random_state=42,
#     n_jobs=-1
# )

# perm_importance = pd.DataFrame({
#     "feature": feature_names,
#     "importance": result.importances_mean,
#     "std": result.importances_std
# }).sort_values("importance", ascending=False)

# print(perm_importance)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    4.6s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:   23.8s
[Parallel(n_jobs=8)]: Done 300 out of 300 | elapsed:   37.9s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


In [ ]:
from sklearn.inspection import permutation_importance

# Random subset of your test pixels
rng = np.random.default_rng(42)

n_samples = min(1000000, len(X_test))
idx = rng.choice(len(X_test), size=n_samples, replace=False)

# Instead of using the full test set, we extract 100,000 rows

X_perm = X_test[idx]
y_perm = y_test[idx]

result = permutation_importance(
    rf,
    X_perm,
    y_perm,
    n_repeats=3,
    random_state=42,
    n_jobs=1
)